# FPGA ML Pollution Prediction Project
### Stage: Window Dataset Generation
### Target FPGA: Basys-3 (Artix-7)
### Task: Predict PM2.5(t+1)
### Window size: 6 hours
### Input dimension: 54
### Output dimension: 1

In [1]:
import pandas as pd
import numpy as np

DATA_PATH = "data/clean_air_quality.csv"

df = pd.read_csv(DATA_PATH, parse_dates=["Timestamp"])

print("Dataset shape:", df.shape)
print(df.head())

Dataset shape: (20816, 10)
            Timestamp    PM10   PM25    PM1    O3    CO    NO   NO2    NOx  \
0 2020-01-13 18:00:00   85.03  52.10  23.15   9.0  0.95  5.61  6.72  10.79   
1 2020-01-13 21:00:00  115.50  84.18  46.73  11.0  1.18  2.48  5.91   8.39   
2 2020-01-13 22:00:00  118.78  86.38  47.35   9.0  1.25  2.53  5.97   8.50   
3 2020-01-13 23:00:00   98.78  71.60  38.78   9.0  1.27  2.48  6.00   8.48   
4 2020-01-14 00:00:00  115.60  86.10  48.50   7.0  1.27  2.34  5.98   8.32   

     CO2  
0  467.7  
1  255.4  
2  223.4  
3  204.9  
4  197.7  


In [2]:
pollutant_cols = [
    "PM10","PM25","PM1",
    "O3","CO","NO","NO2","NOx","CO2"
]

timestamps = df["Timestamp"].values
features = df[pollutant_cols].values.astype(np.float32)

pm25_index = pollutant_cols.index("PM25")

print("Feature matrix shape:", features.shape)
print("PM25 index:", pm25_index)

Feature matrix shape: (20816, 9)
PM25 index: 1


In [3]:
W = 6

X_windows = []
y_targets = []

dropped_windows = 0

for i in range(W, len(df)-1):

    valid = True
    
    # check window continuity
    for j in range(i-W, i):
        if timestamps[j+1] - timestamps[j] != np.timedelta64(1, 'h'):
            valid = False
            break
    
    # check prediction step continuity
    if timestamps[i+1] - timestamps[i] != np.timedelta64(1, 'h'):
        valid = False
    
    if not valid:
        dropped_windows += 1
        continue

    window = features[i-W:i].flatten()
    target = features[i+1][pm25_index]

    X_windows.append(window)
    y_targets.append(target)

X_windows = np.array(X_windows, dtype=np.float32)
y_targets = np.array(y_targets, dtype=np.float32)

print("Windows generated:", len(X_windows))
print("Dropped windows:", dropped_windows)

Windows generated: 20323
Dropped windows: 486


In [4]:
print("X_windows shape:", X_windows.shape)
print("y_targets shape:", y_targets.shape)

X_windows shape: (20323, 54)
y_targets shape: (20323,)


In [5]:
np.save("data/X_windows.npy", X_windows)
np.save("data/y_targets.npy", y_targets)

print("Saved:")
print("data/X_windows.npy")
print("data/y_targets.npy")

Saved:
data/X_windows.npy
data/y_targets.npy


In [6]:
print("Feature dtype:", X_windows.dtype)
print("Target dtype:", y_targets.dtype)

print("First window sample:")
print(X_windows[0][:10])

print("First target value:")
print(y_targets[0])

Feature dtype: float32
Target dtype: float32
First window sample:
[115.5   84.18  46.73  11.     1.18   2.48   5.91   8.39 255.4  118.78]
First target value:
47.23
